In [2]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import pandas as pd
import os
from tqdm.auto import tqdm
import numpy as np

from qick import *
from qick.pyro import make_proxy
from qick import QickConfig
from qick.asm_v2 import QickSpan, QickSweep1D
import Pyro4

Pyro4.config.SERIALIZER = "pickle"
Pyro4.config.PICKLE_PROTOCOL_VERSION = 4

ns_host = "192.168.10.82"
ns_port = 8888
proxy_name = "myqick"

soc, soccfg = make_proxy(ns_host=ns_host, ns_port=ns_port, proxy_name=proxy_name)
print(soccfg)

Pyro.NameServer PYRO:Pyro.NameServer@0.0.0.0:8888
myqick PYRO:obj_4ce8cf0b4edf4470a598e5d1b1c2bc45@192.168.10.82:44717
QICK running on ZCU216, software version 0.2.381

Firmware configuration (built Tue Sep 10 16:13:40 2024):

	Global clocks (MHz): tProc dispatcher timing 430.080, RF reference 245.760
	Groups of related clocks: [tProc timing clock, DAC tile 0, DAC tile 1, DAC tile 3], [DAC tile 2], [ADC tile 2]

	16 signal generator channels:
	0:	axis_signal_gen_v6 - fs=9584.640 Msps, fabric=599.040 MHz
		envelope memory: 16384 complex samples (1.709 us)
		32-bit DDS, range=9584.640 MHz
		DAC tile 2, blk 0 is 0_230 on JHC3, or QICK box DAC port 8
	1:	axis_signal_gen_v6 - fs=9584.640 Msps, fabric=599.040 MHz
		envelope memory: 4096 complex samples (0.427 us)
		32-bit DDS, range=9584.640 MHz
		DAC tile 2, blk 1 is 1_230 on JHC4, or QICK box DAC port 9
	2:	axis_signal_gen_v6 - fs=9584.640 Msps, fabric=599.040 MHz
		envelope memory: 8192 complex samples (0.855 us)
		32-bit DDS, range=9584.

In [1]:
from qick_workspace.scrip.s008_T1_ge import T1
from qick_workspace.scrip.s007_SpinEcho_ge import SpinEcho
from qick_workspace.scrip.s006_Ramsey_ge import Ramsey
from qick_workspace.tools.system_tool import ExperimentConfig
from rshield import config_list

config_all = ExperimentConfig(config_list)


In [4]:
config_all.update('res.ro_length',5)
config_all.update('res.res_gain_ge',0.1)
config_all.update('relax_delay',50)

In [17]:
temperature = 210
avg = 50
from time import sleep
sleep(10*60)

T2_time=2
T1_time=3

In [18]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import numpy as np


# --- 參數設定 ---

csv_dir = "Rshield\mic"
filename = f"{csv_dir}/qubit_results_{temperature}mK.csv"
qubits = ["Q1", "Q3", "Q4"]

# 自動建立資料夾
if not os.path.exists(csv_dir):
    os.makedirs(csv_dir)



# 關閉繪圖視窗
plt.ioff()
pbar = tqdm(qubits)

try:
    for q_name in pbar:
        pbar.set_description(f"Measuring {q_name}")
        
        # 每次重新取得 config 確保環境乾淨
        config_all = ExperimentConfig(config_list)
        run_cfg = config_all.get_qubit(q_name)

        # # --- Ramsey ($T_{2}^*$) ---
        # run_cfg.update([
        #     ("steps", 100), 
        #     ("wait_time", QickSweep1D("waitloop", 0.0, 50)), 
        #     ("ramsey_freq", 0.2)
        # ])
        # t2r = Ramsey(soc, soccfg, run_cfg)
        # t2rfit, t2rerror = t2r.run(avg)
        # t2r.saveLabber(qb_idx=q_name, config_all=config_all, title=f'{temperature}mK')
        # plt.close('all')

        # --- Echo ($T_{2e}$) ---
        run_cfg.update([
            ("steps", 100), 
            ("wait_time", QickSweep1D("waitloop", 0.0, T2_time)), 
            ("ramsey_freq", 2)
        ])
        t2e = SpinEcho(soc, soccfg, run_cfg)
        t2efit, t2eerror = t2e.run(avg)
        t2e.saveLabber(qb_idx=q_name, config_all=config_all, title=f'{temperature}mK')
        plt.close('all')

        # --- T1 ---
        run_cfg.update([
            ("steps", 100), 
            ("wait_time", QickSweep1D("waitloop", 0.0, T1_time)),
        ])
        t1 = T1(soc, soccfg, run_cfg)
        t1fit, t1error = t1.run(avg)
        t1.saveLabber(qb_idx=q_name, config_all=config_all, title=f'{temperature}mK')
        plt.close('all')

        # --- 數據整理 ---
        # t2r_val, t2r_err = t2rfit[3], np.sqrt(np.diag(t2rerror))[3][3]
        t2e_val, t2e_err = t2efit[3], np.sqrt(np.diag(t2eerror))[3][3]
        t1_val, t1_err = t1fit[2], np.sqrt(np.diag(t1error))[2][2]

        data = {
            'qubit': q_name,
            't1_time': t1_val,
            't1_err': t1_err,
            # 't2r_time': t2r_val,
            # 't2r_err': t2r_err,
            't2e_time': t2e_val,
            't2e_err': t2e_err,
            'temp_mK': temperature,
            'timestamp': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
        }

        # --- 即時存檔 ---
        df_single = pd.DataFrame([data])
        file_exists = os.path.isfile(filename)
        df_single.to_csv(filename, mode='a', index=False, header=not file_exists)

        # 輸出進度，使用 .2f 格式化讓數值更易讀
        tqdm.write(f"✅ {q_name} | T1: {t1_val:.2f}±{t1_err:.2f} | T2e: {t2e_val:.2f}±{t2e_err:.2f} us")

except KeyboardInterrupt:
    tqdm.write("\n[!] 偵測到手動中斷。正在保留已寫入的數據...")
except Exception as e:
    tqdm.write(f"\n[X] 執行過程中發生錯誤: {e}")
finally:
    plt.ion()
    tqdm.write("\n量測任務結束。")

Data saved to D:\Labber_Data\Jay\purcell_tmon\Rshield\temperature\2026\03\Data_0331\s008_T1_ge_Q4_210mK_001
✅ Q4 | T1: 1.35±0.88 | T2e: 0.78±0.61 us

量測任務結束。


# long time  monitror

In [12]:
# import os
# import pandas as pd
# import matplotlib.pyplot as plt
# from tqdm.auto import tqdm
# import numpy as np
# import time

# # --- settings ---
# avg = 40
# csv_dir = "T1_2"
# filename = f"{csv_dir}/qubit_monitor_data.csv"
# qubits = ["Q1", "Q3", "Q4"]

# if not os.path.exists(csv_dir):
#     os.makedirs(csv_dir)

# plt.ioff()
# round_idx = 1

# try:
#     while True:
#         tqdm.write(f"\n🚀 starting monitoring round {round_idx} at {pd.Timestamp.now().strftime('%H:%M:%S')}")
        
#         # internal qubit loop
#         pbar = tqdm(qubits, leave=False)
#         for q_name in pbar:
#             pbar.set_description(f"measuring {q_name}")

#             config_all = ExperimentConfig(config_list)
#             run_cfg = config_all.get_qubit(q_name)

#             # --- T2e measurement ---
#             # using spin echo for T2e monitoring
#             run_cfg.update([
#                 ("steps", 100),
#                 ("wait_time", QickSweep1D("waitloop", 0.0, 150)), 
#                 ("ramsey_freq", 0.05)
#             ])
#             t2e = SpinEcho(soc, soccfg, run_cfg)
#             t2efit, t2eerror = t2e.run(avg)
#             t2e.saveLabber(qb_idx=q_name, config_all=config_all, title='monitor_echo')
            
#             # --- T1 measurement ---
#             # monitoring relaxation time T1
#             run_cfg.update([
#                 ("steps", 100), 
#                 ("wait_time", QickSweep1D("waitloop", 0.0, 300))
#             ])
#             t1 = T1(soc, soccfg, run_cfg)
#             t1fit, t1error = t1.run(avg)
#             t1.saveLabber(qb_idx=q_name, config_all=config_all, title='monitor_t1')
            
#             plt.close('all')

#             # --- data processing ---
#             # extracting parameters from fitting results
#             t2e_val, t2e_err = t2efit[3], np.sqrt(np.diag(t2eerror))[3][3]
#             t1_val, t1_err = t1fit[2], np.sqrt(np.diag(t1error))[2][2]

#             data = {
#                 'round': round_idx,
#                 'qubit': q_name,
#                 't1_time': t1_val,
#                 't1_err': t1_err,
#                 't2e_time': t2e_val,
#                 't2e_err': t2e_err,
#                 'timestamp': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
#             }

#             # save immediately to avoid data loss
#             df_single = pd.DataFrame([data])
#             file_exists = os.path.isfile(filename)
#             df_single.to_csv(filename, mode='a', index=False, header=not file_exists)

#             tqdm.write(f"✅ {q_name} | $T_1$: {t1_val:.2f} | $T_{{2e}}$: {t2e_val:.2f} us")

#         round_idx += 1
#         # optional: add a small sleep here if you want to pace the rounds
#         # time.sleep(60) 

# except KeyboardInterrupt:
#     tqdm.write("\n[!] keyboard interrupt detected stopping monitor and saving state")
# except Exception as e:
#     tqdm.write(f"\n[X] error during measurement: {e}")
# finally:
#     plt.ion()
#     tqdm.write(f"\nmonitoring session ended after {round_idx-1} rounds")

In [ ]:
    
    def body(self):
        self.mathi(self.q_rp,self.r_gain,self.r_gain2,"+",0)
        
        self.pulse(ch=self.cfg["qubit_ch"])  #play probe pulse
        self.sync_all(self.us2cycles(0.05))
        
        self.measure(pulse_ch=self.cfg["res_ch"], 
             adcs=[0,1],
             adc_trig_offset=self.cfg["adc_trig_offset"])
        
        self.wait_all(200) # pause until 200 clocks past the end of the readout window
        self.read(0,0,"lower",2)
        self.read(0,0,"upper",3)
        self.condj(0,2,'<',self.r_thresh,'after_reset')

        self.regwi(self.q_rp, self.r_gain, self.cfg["pi_gain"])  #pi pulse qubit
        self.pulse(ch=self.cfg["qubit_ch"], t=0)

        self.label('after_reset')
        self.sync_all(self.us2cycles(1)) # align channels and wait 50ns

        #trigger measurement, play measurement pulse, wait for qubit to relax
        self.measure(pulse_ch=self.cfg["res_ch"], 
             adcs=[0,1],
             adc_trig_offset=self.cfg["adc_trig_offset"],
             wait=True,
             syncdelay=self.us2cycles(self.cfg["relax_delay"]))

    def update(self):
        self.mathi(self.q_rp, self.r_gain2, self.r_gain2, '+', self.cfg["step"]) # update frequency list index